# Member 4: Model Training & 5-Fold Validation Pipeline
**Project 7**: SBERT Sentence Embeddings + Cross-Project Setting + Random Split

### Member 4 Responsibilities (Strict Protocol):
1. **Development Data ONLY**: Test sets are **strictly isolated** and never touched.
2. **Simple Baseline**: Median dummy predictor benchmark on development data.
3. **3-Layer 1D CNN**: Architecture specified in Project Guidelines Section 3.1.
4. **5-Fold Cross-Validation**: 4 folds for training, 1 fold for validation across development data.
5. **Model Serialization**: Train the final model on full development data and save checkpoints to `models/` for Member 5 to load without retraining.

In [1]:
import os
import glob
import json
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, median_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

# Project paths
ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(DATA_DIR, 'splits')
PAIRS_FILE = os.path.join(ROOT, 'config', 'project_pairs.csv')
MODELS_DIR = os.path.join(ROOT, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)

# Load the 39 assigned source-target pairs
pairs_df = pd.read_csv(PAIRS_FILE)

if len(pairs_df) != 39:
    raise ValueError(f"Expected 39 pairs, found {len(pairs_df)}")

print(f'Total project pairs: {len(pairs_df)}')

Using Device: cpu
Total project pairs: 39


## 1. Model Architecture: 3-Layer 1D CNN (Section 3.1)

In [2]:
class StoryPointCNN(nn.Module):
    """
    3-Layer 1D CNN regressor for SBERT sentence embeddings (384 dimensions).
    Conforms directly to Project Guidelines Section 3.1.
    """
    def __init__(self, input_dim=384, dropout=0.2):
        super(StoryPointCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)  # 384 -> 192

        self.conv2 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)  # 192 -> 96

        self.conv3 = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(2)  # 96 -> 48

        self.fc = nn.Sequential(
            nn.Linear(16 * (input_dim // 8), 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x).squeeze(-1)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)

def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, in loader:
            preds.extend(model(X_b.to(device)).cpu().numpy())
    return np.array(preds)

## 2. Member 4 Core Workflow: 5-Fold Cross-Validation on Development Data Only
For each project pair:
* We isolate the development data (all issues except the held-out project).
* We run **5-Fold Cross-Validation** exclusively on development data.
* We fit the final model on 100% of the development data.
* We save the model weights and JSON configuration to `models/` for Member 5.

In [3]:
EPOCHS = 5
BATCH_SIZE = 512
LR = 1e-3

cv_summary_records = []

for idx, pair_row in pairs_df.iterrows():

    pair_name = pair_row['pair']
    source_id = pair_row['source_project_id']
    target_id = pair_row['target_project_id']

    print(f'\n========== {pair_name}: {source_id} -> {target_id} ==========')

    pair_dir = os.path.join(SPLITS_DIR, pair_name)
    embedding_dir = os.path.join(pair_dir, 'embeddings')

    # Load ONLY source-project development data
    X_dev = np.load(
        os.path.join(embedding_dir, 'development_embeddings.npy')
    )

    dev_df = pd.read_csv(
        os.path.join(embedding_dir, 'development_metadata.csv')
    )

    # Validate project identity
    if not (dev_df['project_id'] == source_id).all():
        raise ValueError(f'{pair_name}: Incorrect development project.')

    y_dev = dev_df['story_points'].to_numpy(dtype=np.float32)

    if len(X_dev) != len(y_dev):
        raise ValueError(f'{pair_name}: Embedding-label mismatch.')

    if len(y_dev) < 5:
        raise ValueError(f'{pair_name}: Insufficient records for 5-fold CV.')

    print('Development records:', len(y_dev))
    print('Embedding shape:', X_dev.shape)

    # ------------------------------------------
    # 1. Five-fold cross-validation
    # ------------------------------------------

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        kf.split(X_dev), start=1
    ):

        X_train, X_val = X_dev[train_idx], X_dev[val_idx]
        y_train, y_val = y_dev[train_idx], y_dev[val_idx]

        train_loader = DataLoader(
            TensorDataset(
                torch.tensor(X_train, dtype=torch.float32).unsqueeze(1),
                torch.tensor(y_train, dtype=torch.float32)
            ),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader = DataLoader(
            TensorDataset(
                torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
            ),
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        # CNN
        cv_model = StoryPointCNN(input_dim=384).to(device)

        optimizer = torch.optim.Adam(
            cv_model.parameters(),
            lr=LR,
            weight_decay=1e-4
        )

        criterion = nn.L1Loss()

        for epoch in range(EPOCHS):
            train_epoch(
                cv_model, train_loader, optimizer, criterion, device
            )

        val_preds = predict(cv_model, val_loader, device)

        val_mae = mean_absolute_error(y_val, val_preds)
        val_mdae = median_absolute_error(y_val, val_preds)

        # Median baseline calculated ONLY from fold training labels
        train_median = float(np.median(y_train))
        baseline_preds = np.full(len(y_val), train_median)

        baseline_mae = mean_absolute_error(y_val, baseline_preds)
        baseline_mdae = median_absolute_error(y_val, baseline_preds)

        fold_metrics.append({
            'fold': fold,
            'cnn_mae': round(float(val_mae), 4),
            'cnn_mdae': round(float(val_mdae), 4),
            'baseline_mae': round(float(baseline_mae), 4),
            'baseline_mdae': round(float(baseline_mdae), 4)
        })

        print(
            f'Fold {fold}: '
            f'CNN MAE={val_mae:.4f}, '
            f'CNN MdAE={val_mdae:.4f}, '
            f'Baseline MAE={baseline_mae:.4f}, '
            f'Baseline MdAE={baseline_mdae:.4f}'
        )

    # ------------------------------------------
    # 2. Aggregate CV results
    # ------------------------------------------

    cv_mean_mae = np.mean([m['cnn_mae'] for m in fold_metrics])
    cv_std_mae = np.std([m['cnn_mae'] for m in fold_metrics])

    cv_mean_mdae = np.mean([m['cnn_mdae'] for m in fold_metrics])
    cv_std_mdae = np.std([m['cnn_mdae'] for m in fold_metrics])

    baseline_mean_mae = np.mean(
        [m['baseline_mae'] for m in fold_metrics]
    )

    baseline_mean_mdae = np.mean(
        [m['baseline_mdae'] for m in fold_metrics]
    )

    # ------------------------------------------
    # 3. Train final model on ALL development data
    # ------------------------------------------

    full_train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_dev, dtype=torch.float32).unsqueeze(1),
            torch.tensor(y_dev, dtype=torch.float32)
        ),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    final_model = StoryPointCNN(input_dim=384).to(device)

    optimizer = torch.optim.Adam(
        final_model.parameters(),
        lr=LR,
        weight_decay=1e-4
    )

    criterion = nn.L1Loss()

    for epoch in range(EPOCHS):
        train_epoch(
            final_model, full_train_loader, optimizer, criterion, device
        )

    # ------------------------------------------
    # 4. Save model and configuration
    # ------------------------------------------

    pair_model_dir = os.path.join(MODELS_DIR, pair_name)
    os.makedirs(pair_model_dir, exist_ok=True)

    torch.save(
        final_model.state_dict(),
        os.path.join(pair_model_dir, 'story_point_cnn.pt')
    )

    config_payload = {
        'project': 'Project 7',
        'setting': 'SBERT + Cross-Project + Random Split',
        'pair': pair_name,
        'source_project_id': source_id,
        'target_project_id': target_id,
        'development_records': len(y_dev),
        'model_architecture': 'StoryPointCNN (3-Layer 1D CNN)',
        'input_dimension': 384,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'random_seed': 42,
        'baseline': {
            'mean_cv_mae': round(float(baseline_mean_mae), 4),
            'mean_cv_mdae': round(float(baseline_mean_mdae), 4)
        },
        'cv_validation_results': {
            'folds': fold_metrics,
            'mean_mae': round(float(cv_mean_mae), 4),
            'std_mae': round(float(cv_std_mae), 4),
            'mean_mdae': round(float(cv_mean_mdae), 4),
            'std_mdae': round(float(cv_std_mdae), 4)
        },
        'saved_model_file': 'story_point_cnn.pt'
    }

    with open(
        os.path.join(pair_model_dir, 'model_config.json'), 'w'
    ) as f:
        json.dump(config_payload, f, indent=4)

    cv_summary_records.append({
        'pair': pair_name,
        'source_project_id': source_id,
        'target_project_id': target_id,
        'development_records': len(y_dev),
        'baseline_mae': round(float(baseline_mean_mae), 4),
        'baseline_mdae': round(float(baseline_mean_mdae), 4),
        'cv_mean_mae': round(float(cv_mean_mae), 4),
        'cv_std_mae': round(float(cv_std_mae), 4),
        'cv_mean_mdae': round(float(cv_mean_mdae), 4),
        'cv_std_mdae': round(float(cv_std_mdae), 4)
    })

    print(
        f'{pair_name}: {source_id} -> {target_id} | '
        f'Baseline MdAE: {baseline_mean_mdae:.2f} | '
        f'CNN MdAE: {cv_mean_mdae:.2f}'
    )

# ------------------------------------------
# 5. Save summary
# ------------------------------------------

summary_df = pd.DataFrame(cv_summary_records)

summary_df.to_csv(
    os.path.join(MODELS_DIR, 'cv_summary_39_pairs.csv'),
    index=False
)

print('\nMEMBER 4 TRAINING COMPLETED FOR ALL 39 PAIRS.')


========== pair_01: XD -> MESOS ==========
Development records: 3537
Embedding shape: (3537, 384)


Fold 1: CNN MAE=2.1252, CNN MdAE=1.4612, Baseline MAE=2.0537, Baseline MdAE=2.0000
Fold 2: CNN MAE=2.1853, CNN MdAE=1.6565, Baseline MAE=2.1186, Baseline MdAE=2.0000
Fold 3: CNN MAE=2.1652, CNN MdAE=1.4557, Baseline MAE=2.0382, Baseline MdAE=2.0000
Fold 4: CNN MAE=2.0257, CNN MdAE=1.3452, Baseline MAE=1.9120, Baseline MdAE=2.0000
Fold 5: CNN MAE=1.9732, CNN MdAE=1.4336, Baseline MAE=1.9052, Baseline MdAE=2.0000
pair_01: XD -> MESOS | Baseline MdAE: 2.00 | CNN MdAE: 1.47

========== pair_02: USERGRID -> MXNET ==========
Development records: 482
Embedding shape: (482, 384)
Fold 1: CNN MAE=2.2814, CNN MdAE=2.3224, Baseline MAE=0.8454, Baseline MdAE=0.0000
Fold 2: CNN MAE=2.3185, CNN MdAE=2.6584, Baseline MAE=1.0000, Baseline MdAE=1.0000
Fold 3: CNN MAE=2.6755, CNN MdAE=2.8315, Baseline MAE=0.9896, Baseline MdAE=0.5000
Fold 4: CNN MAE=2.3485, CNN MdAE=2.5044, Baseline MAE=0.9271, Baseline MdAE=1.0000
Fold 5: CNN MAE=2.5157, CNN MdAE=2.5572, Baseline MAE=0.6667, Baseline MdAE=0.0000
pair_02

## 3. Member 4 Validation Results Summary Table (Development Data Only)
This is the official table of validation results across all 39 development sets.

In [4]:
print('========================================================================')
print('MEMBER 4: 5-FOLD CROSS-VALIDATION SUMMARY (DEVELOPMENT DATA ONLY)')
print('========================================================================')
print(f'Overall Baseline Mean MdAE:  {summary_df["baseline_mdae"].mean():.4f} (+/- {summary_df["baseline_mdae"].std():.4f})')
print(f'Overall 3-Layer CNN Mean MdAE: {summary_df["cv_mean_mdae"].mean():.4f} (+/- {summary_df["cv_mean_mdae"].std():.4f})')
reduction = (1 - summary_df['cv_mean_mdae'].mean() / summary_df['baseline_mdae'].mean()) * 100
print(f'Average Error Reduction across Development Sets: {reduction:.2f}%')
print('========================================================================')
summary_df

MEMBER 4: 5-FOLD CROSS-VALIDATION SUMMARY (DEVELOPMENT DATA ONLY)
Overall Baseline Mean MdAE:  1.3987 (+/- 0.8269)
Overall 3-Layer CNN Mean MdAE: 2.0659 (+/- 1.3730)
Average Error Reduction across Development Sets: -47.70%


,pair,source_project_id,target_project_id,development_records,baseline_mae,baseline_mdae,cv_mean_mae,cv_std_mae,cv_mean_mdae,cv_std_mdae
0,pair_01,XD,MESOS,3537,2.0055,2.00,2.0949,0.0820,1.4704,0.1019
1,pair_02,USERGRID,MXNET,482,0.8858,0.50,2.4279,0.1475,2.5748,0.1684
2,pair_03,TIDOC,USERGRID,1279,3.0393,2.00,3.2728,0.5028,1.6443,0.3716
3,pair_04,TIMOB,ALOY,4409,2.9797,2.00,3.1147,0.1154,1.5197,0.3708
4,pair_05,JSWCLOUD,STL,295,2.2186,1.70,3.7115,0.5714,3.2260,0.8011
5,pair_06,NEXUS,MXNET,1804,0.9942,0.50,1.1206,0.0836,0.7902,0.1018
6,pair_07,APIKIT,MULE,465,1.8753,1.80,3.0826,0.4209,2.2055,0.5471
7,pair_08,DAEMON,TISTUD,239,5.1019,3.00,7.4992,1.0427,5.1050,1.1512
8,pair_09,CLI,TIMOB,357,1.7945,2.00,3.0718,0.2184,2.4029,0.1443
9,pair_10,TIDOC,APSTUD,1279,3.0393,2.00,3.2981,0.2743,1.3793,0.2546


## 4. Package Models for Member 5
This creates `models_for_member_5.zip` containing all trained `.pt` models and `model_config.json` files so Member 5 can load them and run evaluation without retraining.

In [5]:
zip_output_path = os.path.join(ROOT, 'models_for_member_5')
shutil.make_archive(zip_output_path, 'zip', MODELS_DIR)
zip_file = zip_output_path + '.zip'
print(f'SUCCESS! Created handoff zip file for Member 5: {zip_file}')
print(f'Size: {os.path.getsize(zip_file) / (1024 * 1024):.2f} MB')
print('You can download this zip file directly from Kaggle and send it to Member 5.')

SUCCESS! Created handoff zip file for Member 5: c:\Users\user\Desktop\Study Material\S7\NLP\Assignment\sbert-story-point-estimation\models_for_member_5.zip
Size: 8.02 MB
You can download this zip file directly from Kaggle and send it to Member 5.
